In [13]:
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal, TypedDict
from langgraph.graph import StateGraph, START, END

In [10]:
model=0

In [4]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="sentiment of the review")

In [ ]:
structured_model=model.with_structured_output(SentimentSchema)

In [12]:
class ReviewState(TypedDict):
    review:str
    sentiment: Literal["positive", "negetive"]
    diagnosis: dict
    response: str

In [20]:
def find_sentiment(state: ReviewState):
    prompt= f"for the following review find out the sentiment \n {state["review"]}"
    sentiment = structured_model.invoke(prompt).sentiment

    return {"sentiment": sentiment}

In [14]:
def check_sentiment(state:ReviewState)-> Literal["positive_response", "run_diagnosis"]:
    if state["sentiment"] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

In [15]:
def positive_response(state:ReviewState):
    prompt=f"write a warm thank-you message in response to this review \n \n {state["review"]}"
    response=model.invoke(prompt)

    return {"response": response}

In [ ]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX","Performance","Bug","Support","Other"] = Field(description="The category of issue mentioned in review")
    tone: Literal["frustrated","disappointed","calm"] = Field(description="The emotional tone expressed by the user")
    urgency: Literal["low","medium","high"] = Field(description="How urgent or critical the issue appears to be")

#structured_model2=model.with_structured_output(DiagnosisSchema)
def run_diagnosis(state:ReviewState):
    prompt=f"""Diagnose this negetive review: \n\n{state["review"]}
    Return issue_type, tone and urgency"""

    response=structured_model2.invoke(prompt)
    return {"diagnosis": response.model_dump()}


In [18]:
def negetive_response(state:ReviewState):
    prompt=f"""The user had a '{state["diagnosis"]["issue_type"]}' issue,
    sounded '{state["diagnosis"]["tone"]}' and marked urgency as 
    '{state["diagnosis"]["urgency"]}'. Write an empatheic, helpful resolution message"""

    response = model.invoke(prompt)
    return {"response": response}

In [25]:
mygraph = StateGraph(ReviewState)

mygraph.add_node("find_sentiment", find_sentiment)
mygraph.add_node("positive_response", positive_response)
mygraph.add_node("run_diagnosis", run_diagnosis)
mygraph.add_node("negetive_response", negetive_response)

mygraph.add_edge(START, "find_sentiment")
mygraph.add_conditional_edges("find_sentiment", check_sentiment)
mygraph.add_edge("positive_response", END)
mygraph.add_edge("run_diagnosis", "negetive_response")
mygraph.add_edge("negetive_response", END)

workflow=mygraph.compile()